In [20]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from nltk.tokenize import TreebankWordTokenizer

from scipy.sparse import hstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
)

from src.config import (
    MIN_COUNT,
    SEEDS,
    THRESHOLD,
    RESULTS_DIR,
    TOKENIZER,
    load_split,
    evaluate,
    LOWERCASE,
    save_predictions,
)

In [21]:
train_df = load_split("train")
val_df = load_split("validation")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)

Train: (3668, 4)
Validation: (408, 4)


In [22]:
print("Tokenizer:", TOKENIZER)
print("Lowercase:", LOWERCASE)
print("Minimum count:", MIN_COUNT)
print("Seeds:", SEEDS)
print("Decision threshold:", THRESHOLD)

Tokenizer: treebank
Lowercase: True
Minimum count: 2
Seeds: [42, 43, 44]
Decision threshold: 0.5


In [23]:
treebank = TreebankWordTokenizer()

def tokenize(text):
    if LOWERCASE:
        text = text.lower()

    return treebank.tokenize(text)

In [24]:
y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()

majority_class = train_df["label"].mode()[0]

majority_pred = np.full(
    len(val_df),
    majority_class
)

majority_results = evaluate(
    y_val,
    majority_pred
)

print("Majority class:", majority_class)
print(f"Accuracy: {majority_results['accuracy']:.4f}")
print(f"F1:       {majority_results['f1']:.4f}")

Majority class: 1
Accuracy: 0.6838
F1:       0.8122


In [25]:
training_sentences = pd.concat(
    [
        train_df["sentence1"],
        train_df["sentence2"],
    ],
    ignore_index=True,
)

tfidf = TfidfVectorizer(
    tokenizer=tokenize,
    token_pattern=None,
    lowercase=False,
    min_df=MIN_COUNT,
)

tfidf.fit(training_sentences)

print("TF-IDF vocabulary size:", len(tfidf.vocabulary_))

TF-IDF vocabulary size: 9917


In [26]:
train_s1 = tfidf.transform(train_df["sentence1"])
train_s2 = tfidf.transform(train_df["sentence2"])

val_s1 = tfidf.transform(val_df["sentence1"])
val_s2 = tfidf.transform(val_df["sentence2"])

print("Train S1:", train_s1.shape)
print("Train S2:", train_s2.shape)
print("Validation S1:", val_s1.shape)
print("Validation S2:", val_s2.shape)

Train S1: (3668, 9917)
Train S2: (3668, 9917)
Validation S1: (408, 9917)
Validation S2: (408, 9917)


In [27]:
train_diff = abs(train_s1 - train_s2)
val_diff = abs(val_s1 - val_s2)

train_product = train_s1.multiply(train_s2)
val_product = val_s1.multiply(val_s2)

X_train = hstack(
    [train_diff, train_product]
).tocsr()

X_val = hstack(
    [val_diff, val_product]
).tocsr()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

X_train: (3668, 19834)
X_val: (408, 19834)


In [28]:
c_values = [0.01, 0.1, 1.0, 10.0, 100.0]

tuning_results = []

for c in c_values:

    model = LogisticRegression(
        C=c,
        max_iter=1000,
        random_state=SEEDS[0],
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_val)

    scores = evaluate(y_val, preds)

    tuning_results.append({
        "C": c,
        "Accuracy": scores["accuracy"],
        "F1": scores["f1"],
    })

tuning_df = pd.DataFrame(tuning_results)

tuning_df

,C,Accuracy,F1
0,0.01,0.683824,0.812227
1,0.10,0.691176,0.815789
2,1.00,0.740196,0.837423
3,10.00,0.742647,0.824708
4,100.00,0.720588,0.805461


In [29]:
BEST_C = 1.0

lr_model = LogisticRegression(
    C=BEST_C,
    max_iter=1000,
    random_state=SEEDS[0],
)

lr_model.fit(X_train, y_train)

val_prob = lr_model.predict_proba(X_val)[:, 1]

val_pred = (
    val_prob >= THRESHOLD
).astype(int)

val_results = evaluate(
    y_val,
    val_pred
)

print(f"Accuracy: {val_results['accuracy']:.4f}")
print(f"F1:       {val_results['f1']:.4f}")

Accuracy: 0.7402
F1:       0.8374


In [30]:
precision = precision_score(y_val, val_pred)
recall = recall_score(y_val, val_pred)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {val_results['f1']:.4f}")
print(f"Accuracy:  {val_results['accuracy']:.4f}")

Precision: 0.7319
Recall:    0.9785
F1:        0.8374
Accuracy:  0.7402


### Logistic Regression Results

The Logistic Regression model represents each sentence using TF-IDF features.
The TF-IDF vocabulary is fitted only on the training split using the shared
Treebank tokenizer, lowercasing configuration, and `MIN_COUNT=2`.

For each sentence pair, the absolute difference and element-wise product of
the two TF-IDF vectors are concatenated. The absolute difference represents
lexical differences between the sentences, while the element-wise product
captures shared lexical features.

The majority-class baseline achieved a validation accuracy of 0.6838 and an
F1 score of 0.8122.

Logistic Regression regularization values from C=0.01 to C=100 were evaluated
using the validation set. C=1.0 produced the highest F1 score and was selected
for the final model.

The final Logistic Regression classifier achieved a validation accuracy of
0.7402 and an F1 score of 0.8374. It achieved very high recall for the
paraphrase class but produced substantially more false positives than false
negatives, indicating that lexical similarity alone can sometimes make
non-paraphrase sentence pairs appear similar.

In [31]:
test_df = load_split("test")
y_test = test_df["label"].to_numpy()

print("Test:", test_df.shape)

Test: (1725, 4)


In [32]:
test_s1 = tfidf.transform(test_df["sentence1"])
test_s2 = tfidf.transform(test_df["sentence2"])

test_diff = abs(test_s1 - test_s2)
test_product = test_s1.multiply(test_s2)

X_test = hstack(
    [test_diff, test_product]
).tocsr()

print("X_test:", X_test.shape)

X_test: (1725, 19834)


In [33]:
test_probs = lr_model.predict_proba(X_test)[:, 1]

In [34]:
test_scores = save_predictions(
    test_df,
    test_probs,
    RESULTS_DIR / "group2_logreg" / "test_seed42.csv"
)

print(f"Test Accuracy: {test_scores['accuracy']:.4f}")
print(f"Test F1:       {test_scores['f1']:.4f}")

Test Accuracy: 0.7154
Test F1:       0.8164


The final Logistic Regression model achieved:

- Test Accuracy: 0.7154
- Test F1: 0.8164

In [ ]:
test_scores = save_predictions(
    test_df,
    test_probs,
    RESULTS_DIR / "group2_logreg" / "test_seed42.csv"
)